# Introduction

This notebook represents the **next step** in creating **daily sentiment scores** for the companies selected in the **“News Filtering”** notebook, where **NVDA** and **BRK** were chosen based on their **long historical news coverage** and **high volume of valid articles**. The goal of this notebook is to **transform raw news articles** into **daily sentiment scores** that are suitable for **GRU model training**.

# Methodology

## 1. Data Preparation
We begin by **cleaning and preparing the data** for sentiment analysis. This includes **removing any articles that are NaN or contain empty strings**. After this cleaning step, we will **reassess the length of the available news coverage** to decide whether it is appropriate to continue sentiment analysis for the processed company.
Based on the findings from the **“News Filtering”** notebook, **BRK** has no articles with empty or NaN values, so it will be used directly for sentiment analysis. In contrast, **NVDA** contains many articles that are empty or NaN. After cleaning **NVDA’s dataset**, we will evaluate the remaining news coverage and determine whether it is suitable to continue using **NVDA** for sentiment analysis.

## 2. Sentiment Analysis
After data cleaning and preparation, each article will be analyzed using the pre-trained **FinBERT** model. Each article will receive a **positive**, **neutral**, or **negative** label based on its sentiment. These labels will then be **converted into numeric scores**, which will be used to compute the **daily aggregated sentiment**.

## 3. Daily Sentiment Aggregation
Once daily sentiment labels are created, they will be **converted into numeric scores** as follows:
1. Positive = 1.
2. Neutral = 0.
3. Negative = -1.

These scores are then used to compute the **daily aggregated sentiment**. To account for the number of articles published each day, we calculate a **weighted sentiment** by multiplying the **mean daily sentiment** by the **logarithm of the article count**. This ensures that days with more news coverage have a proportionally greater impact on the aggregated sentiment.

## 4. Rolling z-score normalization
To ensure that the aggregated daily sentiment scores are normalized across the years in the dataset, we apply a **rolling z-score** to the **weighted sentiment**. The resulting normalized **weighted sentiment** will then be used as input for **GRU model training**.

---

## Install dependencies

Uncomment the code in the cell below to install the dependencies required for this notebook.

In [1]:
# !pip install transformers torch pandas numpy

## Import dependencies

In [2]:
import pandas as pd
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

/home/user/anaconda3/envs/FinBERT/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Functions for Loading and Saving CSV Files

The functions below handle loading and saving CSV data.
- **load_csv**: Reads a CSV file into a **pandas DataFrame** and prints the number of rows loaded.
- **save_csv**: Writes a **DataFrame** back to a CSV file, and prints a confirmation message.

In [3]:
def load_csv(csv_file_name):
    # Load the given CSV file into a Pandas DataFrame
    df = pd.read_csv(csv_file_name)
    # Print the number of rows loaded
    print(f"Loaded {len(df)} rows from {csv_file_name}")
    return df


def save_csv(df, csv_file_name):
    # Save the given DataFrame to a CSV file using the specified file name
    df.to_csv(csv_file_name, index=False, encoding="utf-8")
    # Print confirmation of saving
    print(f"Saved dataframe to {csv_file_name}")

## Data Cleaning and Preparation

The functions below handle **data cleaning and preparation**:
- **drop_empty_articles**: Removes entries with NaN or empty strings from the specified article column. It prints the number of removed articles and returns the cleaned DataFrame.
- **prepare_news_dataframe**: Loads news data from a CSV file, converts dates to datetime, computes the news coverage in years before and after removing empty articles, and returns the cleaned DataFrame.

In [4]:
def drop_empty_articles(df, article_column):
    # Store the original number of rows
    df_original_length = len(df)

    # Drop rows where the article column is NaN or contains empty strings
    df = df.dropna(subset=[article_column])
    df = df[df[article_column].astype(str).str.strip() != ""]

    # Print number of removed entires
    print(f"Removed {df_original_length - len(df)} empty articles")
    return df.reset_index(drop=True)


def prepare_news_dataframe(csv_file_name, article_column, date_column):
    # Call 'load_csv' function to load data from CSV file into Pandas DataFrame 
    df = load_csv(csv_file_name)

    # Convert Date column into datetime64 format
    df[date_column] = pd.to_datetime(df[date_column], errors="coerce")

    # Calculate the length of news coverage before cleaning
    df_dates_before = df.dropna(subset=[date_column])
    min_date_before = df_dates_before[date_column].min()
    max_date_before = df_dates_before[date_column].max()
    years_before = (max_date_before - min_date_before).days / 365.25
    print(f"Years covered before cleaning: {years_before:.2f}")

    # Call 'drop_empty_articles' function to drop entries with NaN or empty strings
    df = drop_empty_articles(df, article_column)

    # Calculate the length of news coverage after cleaning
    min_date_after = df[date_column].min()
    max_date_after = df[date_column].max()
    years_after = (max_date_after - min_date_after).days / 365.25
    print(f"Years covered after cleaning:  {years_after:.2f}")

    return df


## Load and unload FinBERT model

The functions below handle **loading and unloading the FinBERT model**:
- **load_sentiment_model**: Sets the device to **CPU** by default, and uses **GPU** if available. It then loads the **tokenizer** and **FinBERT model**, moves the model to the device, and prepares it for use. Returns the tokenizer, model, and device.
- **unload_model**: Moves the model back to **CPU**, deletes the model and tokenizer from memory, and clears the GPU cache.

In [5]:
def load_sentiment_model(model_name):
    # Create computation device and set it to 'CPU' by default
    device = "cpu"

    # Check if 'CUDA(GPU)' is available and set the device to it if true 
    if torch.cuda.is_available():
        device = "cuda"
        
    print(f"Using device: {device}")

    # Create tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    # Create passedmodel
    model = AutoModelForSequenceClassification.from_pretrained(model_name)

    # Move model to created above computation device
    model.to(device)
    # Set model to evaluation mode.
    model.eval()

    return tokenizer, model, device

def unload_model(model, tokenizer):
    # Move model to 'CPU'
    model.to("cpu")
    # Delete model
    del model
    # DElete tokenizer
    del tokenizer
    # Empty CUDA cache
    torch.cuda.empty_cache()

## Sentiment Analysis Function

The function below handles **sentiment analysis** for a list of articles. It works in **batches** to process large datasets efficiently.
- For each batch, it analyzes every article and assigns a label: **neutral**, **positive**, or **negative**.
- After each batch, it prints the progress so you can see how many articles have been processed.
- When all articles are processed, it returns a **list of sentiment labels** for the entire dataset.

In [6]:
def analyze_sentiments(texts, tokenizer, model, device, batch_size):

    # Create labels
    label_map = {0: "neutral", 1: "positive", 2: "negative"}

    # List to hold created sentiments for each article
    sentiments = []

    # Save total length of the passed texts
    total = len(texts)

    # Iterate from 0 to 'total' increasing each step by 'batch_size' 
    for i in range(0, total, batch_size):
        # Select 'batch_size' number of articles from the texts
        batch_texts = texts[i:i + batch_size]

        # Tokenize articles from 'batch_texts'
        encoded = tokenizer(
            batch_texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512
        ).to(device)

        # Get sentiments for the batch
        with torch.no_grad():
            logits = model(**encoded).logits
            labels = torch.argmax(logits, dim=1).cpu().numpy()

        # Map numeric values to sentiment labels
        for label in labels:
            sentiments.append(label_map[label])


        # Print progress
        processed = min(i + batch_size, total)
        if i % (batch_size * 10) == 0:
            print(f"Processed {processed}/{total}")
        elif processed == total:
            print(f"Processed final batch: {processed}/{total}")

    return sentiments

## Sentiment Analysis Orchestrator

The function below **runs sentiment analysis on a DataFrame** of articles:
- It first creates a **list of all articles** from the specified column.
- Then it calls **analyze_sentiments** to assign a sentiment label to each article.
- Next, it makes a **copy of the original DataFrame** and adds a new column called **Sentiment** with the results.
- Finally, it returns a DataFrame containing only the **Date** and **Sentiment** columns.

In [7]:
def analyze_sentiment_df(df, tokenizer, model, device, article_column, batch_size):
    # Get all articles from 'article_column'
    articles = df[article_column].astype(str).tolist()

    # Call 'analyze_sentiments' function to create sentiments
    sentiments = analyze_sentiments(
        texts=articles,
        tokenizer=tokenizer,
        model=model,
        device=device,
        batch_size=batch_size
    )

    # Copy the original DataFrame
    df = df.copy()
    # Create new column called 'Sentiments' and populate with the results
    df["Sentiment"] = sentiments

    # Create a new DataFrame with only 'Date' and 'Sentiment' columns
    df_out = df[["Date", "Sentiment"]]
    
    return df_out

## Daily Sentiment Aggregation

The function below handle aggregates of the daily sentiment scores:
- It first creates a map for **positive = 1**, **neutral = 0**, and **negative = -1**.
- It then creates a copy of the input DataFrame and adds a numeric sentiment_score column.
- Converts the date column to a date-only format.
- The data is grouped by date to calculate the **mean sentiment score** for the day, and the **number of articles** published on that day
- A **weighted sentiment score** is computed by multiplying the mean sentiment by the logarithm of the article count.
- Then it uses positive and negative thresholds to assign a **daily sentiment label** for each day.
- The function returns a DataFrame containing the aggregated daily sentiment metrics.

In [8]:
def aggregate_daily_sentiment(df, date_column, sentiment_column, positive_threshold, negative_threshold):
    # Create map for sentiment labels
    sentiment_score_map = {"positive": 1, "neutral": 0, "negative": -1}

    # Create copy of the passed DataFrame
    df = df.copy()
    # Create new column called 'sentiment_score' and populate it with numeric values based on the labels
    df["sentiment_score"] = df[sentiment_column].map(sentiment_score_map)
    
    # df["Date"] = pd.to_datetime(df[date_column]).dt.date

    # Create daily sentiment scores
    daily = (df.groupby("Date").agg(
                                      mean_sentiment=("sentiment_score", "mean"),
                                      article_count=("sentiment_score", "count")
                                    ).reset_index())
    # Calculate weighted sentiment score
    daily["weighted_sentiment"] = (
        daily["mean_sentiment"] * np.log(daily["article_count"] + 1)
    )

    # Give each day a sentiment score based on the daily sentiment score.
    def score_to_label(score):
        if score > positive_threshold:
            return "positive"
        elif score < negative_threshold:
            return "negative"
        return "neutral"

    daily["daily_sentiment"] = daily["mean_sentiment"].apply(score_to_label)

    return daily

## Daily Sentiment Rolling Z-score Normalization

The function below creates a rolling z-score normalization for a given column.
- It first creates a copy of the input DataFrame.
- It calculates the **rolling mean** of the specified column using the given window size and minimum number of periods.
- It calculates the **rolling standard deviation** for the same column.
- A new column is created to store the **rolling z-score** values.
- The rolling z-score is computed by subtracting the rolling mean from each value and dividing by the rolling standard deviation.
- Any missing values in the rolling z-score column are filled with the provided fill value.
- The function returns the updated DataFrame with the new rolling z-score column.

In [9]:
def add_rolling_zscore(df, column, window, min_periods, fill_value):
    # Create copy of the DataFrame
    df = df.copy()

    # Calculate rolling mean score 
    rolling_mean = df[column].rolling(window=window, min_periods=min_periods).mean()

    # Calculate rolling std
    rolling_std = df[column].rolling(window=window, min_periods=min_periods).std()

    # Create a new column called 'original_score_column_name_rolling_z_score'
    z_col = f"{column}_rolling_z_score"

    # Calculate rolling z score
    df[z_col] = (df[column] - rolling_mean) / rolling_std

    # Fill entries with NaN with passed 'fill_value'
    df[z_col] = df[z_col].fillna(fill_value)

    return df

---

## Create FinBERT model for Sentiment Analysis

Here we define the **pretrained sentiment model** to be used and initialize the **tokenizer**, **model**, and **computation device**.

In [10]:
model_name = "ProsusAI/finbert"

tokenizer, model, device = load_sentiment_model(model_name)

Using device: cuda


## NVDA Data Preparation for Sentiment Analysis

In this step, we load, clean, and process the **NVDA** news dataset by removing empty articles and determining the remaining **length of available news coverage in years**.

In [11]:
df_nvda = prepare_news_dataframe("./Inputs/NVDA_news.csv", "Article", "Date")

Loaded 11862 rows from ./Inputs/NVDA_news.csv
Years covered before cleaning: 12.79
Removed 3146 empty articles
Years covered after cleaning:  2.33


After removing articles with **empty strings or NaN values**, the effective coverage of the NVDA news dataset is reduced from **over 12 years to approximately 2 years**.

Although the dataset still contains a good number of articles, they are **concentrated within a short time span**, making the data **unsuitable for GRU model training**, which requires longer and more continuous time series.

As a result, **NVDA will not be used for sentiment analysis**, and no further processing will be performed on this dataset.

## BRK Data Preparation for Sentiment Analysis

In [12]:
df_brk = prepare_news_dataframe("./Inputs/BRK_news.csv", "Article", "Date")

Loaded 8797 rows from ./Inputs/BRK_news.csv
Years covered before cleaning: 14.16
Removed 0 empty articles
Years covered after cleaning:  14.16


The results of the data cleaning and preparation steps show that the **BRK** dataset does not contain any articles with NaN values or empty strings. As a result, no articles were removed, and the total span of news coverage remains the same at **14.16 years**. The next step is to perform **sentiment analysis** on the BRK dataset.

## Run BRK Sentiment Analysis

In [13]:
df_brk_with_sentiment = analyze_sentiment_df(df_brk, tokenizer, model, device, "Article", 32)

unload_model(model, tokenizer)

Processed 32/8797
Processed 352/8797
Processed 672/8797
Processed 992/8797
Processed 1312/8797
Processed 1632/8797
Processed 1952/8797
Processed 2272/8797
Processed 2592/8797
Processed 2912/8797
Processed 3232/8797
Processed 3552/8797
Processed 3872/8797
Processed 4192/8797
Processed 4512/8797
Processed 4832/8797
Processed 5152/8797
Processed 5472/8797
Processed 5792/8797
Processed 6112/8797
Processed 6432/8797
Processed 6752/8797
Processed 7072/8797
Processed 7392/8797
Processed 7712/8797
Processed 8032/8797
Processed 8352/8797
Processed 8672/8797
Processed final batch: 8797/8797


In [14]:
df_brk_with_sentiment.head()

,Date,Sentiment
0,2023-12-16 15:00:00+00:00,neutral
1,2023-12-16 15:00:00+00:00,negative
2,2023-12-16 14:00:00+00:00,positive
3,2023-12-16 14:00:00+00:00,negative
4,2023-12-16 00:00:00+00:00,negative


The resulting DataFrame from the sentiment analysis step contains only the **Date** and **Sentiment** columns, since the textual data is no longer needed. The **Date** and **Sentiment** columns will be used in the next step to calculate **aggregated daily sentiment scores** for each day.

## BRK Sentiment Daily Aggregation, Rolling Normalization

In [15]:
df_brk_daily = aggregate_daily_sentiment(df_brk_with_sentiment, "Date", "Sentiment", 0.1, 0.1)

df_brk_daily = add_rolling_zscore(df_brk_daily, "weighted_sentiment", 60, 1, 0.0)

df_brk_daily.head()

,Date,mean_sentiment,article_count,weighted_sentiment,daily_sentiment,weighted_sentiment_rolling_z_score
0,2009-10-19 00:00:00+00:00,-1.0,1,-0.693147,negative,0.000000
1,2009-11-05 00:00:00+00:00,-1.0,2,-1.098612,negative,-0.707107
2,2009-11-08 00:00:00+00:00,-1.0,2,-1.098612,negative,-0.577350
3,2010-01-25 00:00:00+00:00,-1.0,2,-1.098612,negative,-0.500000
4,2010-02-22 00:00:00+00:00,-1.0,1,-0.693147,negative,1.095445


In this step, we calculated the **weighted sentiment** and its **rolling z-score** for BRK news. The rolling z-score normalizes the daily sentiment values over time, which will be used as input for the GRU model. We save the full DataFrame, including all intermediate columns, in case we need them later.

## Save BRK News Sentiment Analysis

In [16]:
save_csv(df_brk_daily, "./Outputs/BRK_with_aggregated_sentiment_score.csv")

Saved dataframe to ./Outputs/BRK_with_aggregated_sentiment_score.csv
